# 二吸収帯 Iterative Matched Filter による実HISUIメタンプルーム検出

実HISUIシーンそのものに対してIterative Matched Filterを適用

処理の中心は次の反復

1. 現在の背景候補画素から、各吸収帯の背景平均と共分散を推定する
2. 1.6 µm帯・2.3 µm帯でMatched Filterを計算する
3. 高い正のMF応答を示す画素とその周囲を背景候補から除外する
4. 背景統計を再計算し、除外画素がほぼ変化しなくなるまで繰り返す

各帯域のMF出力は背景分布でrobust標準化

$$
Z_b(x,y)=
\frac{\alpha_b(x,y)-\operatorname{median}_{\mathrm{bg}}(\alpha_b)}
{1.4826\,\operatorname{MAD}_{\mathrm{bg}}(\alpha_b)}
$$

2.3 µm帯を主候補生成に使い、1.6 µm帯を空間的な確認に使う。

- Iterative MFの収束履歴
- 1.6 µm帯・2.3 µm帯のMF応答とrobust Z-score
- 2.3 µm候補のうち、1.6 µm帯が近傍で支持する画素・領域
- 二帯域の重なりが偶然以上かを調べるランダムシフト検定

このNotebookのMF出力 $\alpha$ は、まず**メタン標的方向の応答強度**として解釈する。正確なppm推定値とは限らない。


In [ ]:
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import (
    binary_dilation,
    label,
    maximum_filter,
)

np.set_printoptions(precision=5, suppress=True)


## 1. 設定

In [ ]:
# 入力
ROI_CSV = r"E:\refit\all_roi_spectra.csv"
CH4_LUT_CSV = r"E:\refit\CH4b.csv"
OUTPUT_DIR = Path("./dual_window_iterative_mf_real_plume_output")

# MODTRAN・HISUI設定
BACKGROUND_CH4_PPM = 1.8
FWHM_NM = 12.5
WINDOW_16 = (1580.0, 1750.0)
WINDOW_23 = (2100.0, 2450.0)
UAS_MAX_ENHANCEMENT_PPM = 0.5
UAS_STEP_PPM = 0.05

# Iterative MF設定
MAX_ITERATIONS = 12
EXCLUSION_Z_16 = 2.5
EXCLUSION_Z_23 = 2.5
EXCLUSION_DILATION_PIXELS = 2
MIN_BACKGROUND_FRACTION = 0.55
MIN_BACKGROUND_PIXELS = 300
CONVERGENCE_NEW_PIXEL_FRACTION = 2.5e-4
COVARIANCE_SHRINKAGE = 0.08
COVARIANCE_RIDGE_RELATIVE = 1e-8

# 最終候補抽出
DETECTION_Z_16 = 2.0       # 1.6 µm帯は確認用なので少し緩める
DETECTION_Z_23 = 3.0       # 2.3 µm帯を主候補生成に使う
NEIGHBORHOOD_RADIUS = 1    # 1なら3×3近傍を許容
MIN_REGION_PIXELS = 3
TOP_PIXEL_COUNT = 200

# 二帯域の偶然重なりを調べるランダムシフト検定
SHIFT_TEST_TRIALS = 500
SHIFT_TEST_MIN_PIXELS = 5
RANDOM_SEED = 42

# 任意：図に発生源候補を表示する場合、ROI内インデックス(row, col)を指定
SOURCE_YX_INDEX = None  # 例: (100, 85)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


## 2. HISUI ROIスペクトルCSVの読み込み

In [ ]:
def get_wave_columns(df):
    pattern = re.compile(r"^wave_([0-9.]+)nm$")
    pairs = []
    for col in df.columns:
        match = pattern.match(str(col))
        if match is not None:
            pairs.append((col, float(match.group(1))))
    if not pairs:
        raise ValueError(
            f"wave_***nm形式の列が見つかりません。先頭列: {list(df.columns[:10])}"
        )
    pairs.sort(key=lambda item: item[1])
    return [p[0] for p in pairs], np.array([p[1] for p in pairs], dtype=float)


def load_roi_spectra_csv(path):
    df = pd.read_csv(path)
    if "y" not in df.columns or "x" not in df.columns:
        raise ValueError("CSVには y と x 列が必要です。")
    wave_cols, wavelengths = get_wave_columns(df)
    spectra = df[wave_cols].to_numpy(dtype=float)
    return df, wavelengths, spectra


def spectra_to_cube(df, spectra, fill_value=np.nan):
    frame = df.reset_index(drop=True)
    ys = np.sort(frame["y"].unique())
    xs = np.sort(frame["x"].unique())
    y_to_i = {value: i for i, value in enumerate(ys)}
    x_to_i = {value: i for i, value in enumerate(xs)}

    cube = np.full(
        (len(ys), len(xs), spectra.shape[1]),
        fill_value,
        dtype=float,
    )
    for row_i, row in frame.iterrows():
        cube[y_to_i[row["y"]], x_to_i[row["x"]]] = spectra[row_i]
    return cube, ys, xs


def make_valid_pixel_mask(
    cube,
    nodata_values=(0.0, -9999.0),
    require_positive=True,
):
    valid = np.isfinite(cube)
    for value in nodata_values:
        valid &= cube != value
    if require_positive:
        valid &= cube > 0
    return np.all(valid, axis=2)


df, wavelengths, spectra = load_roi_spectra_csv(ROI_CSV)
cube_observed, y_values, x_values = spectra_to_cube(df, spectra)
valid_mask = make_valid_pixel_mask(cube_observed)

print("DataFrame shape:", df.shape)
print("Cube shape:", cube_observed.shape)
print("Wavelength range:", wavelengths[0], "to", wavelengths[-1], "nm")
print("Valid pixels:", int(valid_mask.sum()), "/", valid_mask.size)
